In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Dataset downloaded from the links below:

DIV2K- https://data.vision.ee.ethz.ch/cvl/DIV2K/

Flickr2K - https://www.kaggle.com/datasets/daehoyang/flickr2k



In [ ]:
# verify paths
!ls /content/drive/MyDrive/dl_lab_super_resolution/data/div2k
!ls /content/drive/MyDrive/dl_lab_super_resolution/data/flickr2k

hr  lr_x2  lr_x3  lr_x4
hr  lr_x2  lr_x3  lr_x4


In [ ]:
#setting path and local variables

data_root = "/content/drive/MyDrive/dl_lab_super_resolution/data"
train_hr = f"{data_root}/div2k/hr"
train_lr = f"{data_root}/div2k/lr_x4"
train_hr_flickr = f"{data_root}/flickr2k/hr"
train_lr_flickr = f"{data_root}/flickr2k/lr_x4"
val_hr = f"{data_root}/div2k_official_val/hr"
val_lr = f"{data_root}/div2k_official_val/lr_x4"


×4” (often written as X4, x4, or scale 4) refers to the upscaling factor used in super-resolution (SR)

In our project:

we use ×4 because that’s the official benchmark for DIV2K, Set14, BSD100, Urban100.

swinir and stablesr both have pretrained ×4 checkpoints, so inference is immediate.

training/fine-tuning is straightforward because your dataset already has lr_x4/ prepared.

Tests sets downloaded from here-https://openmmlab.medium.com/awesome-datasets-for-super-resolution-introduction-and-pre-processing-55f8501f8b18


In [ ]:
# unzip set14.zip + bsds100.zip from drive, organize into hr/, and make lr_x4/


import os, zipfile, shutil
from pathlib import Path
from PIL import Image

# ---- paths (edit if your project folder is different)
drive_root_path = "/content/drive/MyDrive/dl_lab_super_resolution"
testsets_root_path = f"{drive_root_path}/data/test_sets"

zip_set14_path = Path(testsets_root_path) / "Set14.zip"
zip_bsd_path   = Path(testsets_root_path) / "BSDS100.zip"  # note: your file is 'BSDS100.zip' per screenshot

out_set14_hr_path  = Path(testsets_root_path) / "set14/hr"
out_set14_lr4_path = Path(testsets_root_path) / "set14/lr_x4"
out_bsd_hr_path    = Path(testsets_root_path) / "bsd100/hr"
out_bsd_lr4_path   = Path(testsets_root_path) / "bsd100/lr_x4"

for p in [out_set14_hr_path, out_set14_lr4_path, out_bsd_hr_path, out_bsd_lr4_path]:
    p.mkdir(parents=True, exist_ok=True)

# ---- helper: unzip to a temp dir
def unzip_to_tmp(zip_file: Path, tmp_dir: Path):
    if not zip_file.exists():
        raise FileNotFoundError(f"missing zip: {zip_file}")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(tmp_dir)
    return tmp_dir

# ---- helper: copy all images from a folder tree into dst
def copy_all_images(src_root: Path, dst_dir: Path):
    cnt = 0
    for fp in src_root.rglob("*"):
        if fp.is_file() and fp.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".tif",".tiff"):
            shutil.copy2(fp, dst_dir / fp.name)
            cnt += 1
    return cnt

# ---- 1) extract set14 hr
tmp_set14_dir = Path("/content/tmp_set14_extract")
unzip_to_tmp(zip_set14_path, tmp_set14_dir)

# set14 zips often contain a top folder named 'Set14' or similar; just grab images from the whole tree
n_set14_hr = copy_all_images(tmp_set14_dir, out_set14_hr_path)

# ---- 2) extract bsd100 hr
tmp_bsd_dir = Path("/content/tmp_bsd_extract")
unzip_to_tmp(zip_bsd_path, tmp_bsd_dir)

n_bsd_hr = copy_all_images(tmp_bsd_dir, out_bsd_hr_path)

# ---- 3) make lr_x4 from hr (bicubic). ensure dims divisible by 4 first
def make_lr_x4_from_hr(hr_dir: Path, lr_dir: Path):
    made = 0
    for f in sorted(hr_dir.glob("*")):
        if f.suffix.lower() not in (".png",".jpg",".jpeg",".bmp",".tif",".tiff"):
            continue
        img = Image.open(f).convert("RGB")
        w, h = img.size
        w4, h4 = w - (w % 4), h - (h % 4)
        if (w4, h4) != (w, h):
            img = img.crop((0, 0, w4, h4))
        lr = img.resize((img.width // 4, img.height // 4), resample=Image.BICUBIC)
        out_name = f.stem
        if "x4" not in out_name.lower():
            out_name = out_name + "x4"
        lr.save(lr_dir / f"{out_name}.png")
        made += 1
    return made

made_set14_lr4 = make_lr_x4_from_hr(out_set14_hr_path, out_set14_lr4_path)
made_bsd_lr4   = make_lr_x4_from_hr(out_bsd_hr_path,   out_bsd_lr4_path)

# ---- 4) sanity report
def count_imgs(d: Path):
    return len([f for f in d.glob("*") if f.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".tif",".tiff")])

print("done.")
print(f"set14  hr: {count_imgs(out_set14_hr_path)}  | lr_x4: {count_imgs(out_set14_lr4_path)}  (expected ~14)")
print(f"bsd100 hr: {count_imgs(out_bsd_hr_path)}    | lr_x4: {count_imgs(out_bsd_lr4_path)}    (expected ~100)")

# show a couple of examples so you can plug paths into eval scripts
def example_paths(dir_path: Path, k=3):
    return [str(p) for p in sorted(dir_path.glob('*.png'))[:k]]

print("\nexamples:")
print("set14 lr_x4:", example_paths(out_set14_lr4_path))
print("bsd100 lr_x4:", example_paths(out_bsd_lr4_path))


done.
set14  hr: 14  | lr_x4: 14  (expected ~14)
bsd100 hr: 100    | lr_x4: 100    (expected ~100)

examples:
set14 lr_x4: ['/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/set14/lr_x4/baboonx4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/set14/lr_x4/barbarax4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/set14/lr_x4/bridgex4.png']
bsd100 lr_x4: ['/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/bsd100/lr_x4/101085x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/bsd100/lr_x4/101087x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/bsd100/lr_x4/102061x4.png']


In [ ]:
# unzip urban100.zip from drive, organize into hr/, and make lr_x4/


import os, zipfile, shutil
from pathlib import Path
from PIL import Image

# ---- paths (edit if your project folder is different)
drive_root_path = "/content/drive/MyDrive/dl_lab_super_resolution"
testsets_root_path = f"{drive_root_path}/data/test_sets"

# try to find the urban100 zip automatically if you aren't sure of the exact name
def find_urban_zip(search_dir: Path):
    cand = []
    for p in search_dir.glob("*.zip"):
        if "urban100" in p.name.lower():
            cand.append(p)
    if cand:
        # prefer exact 'Urban100.zip' if present
        for p in cand:
            if p.name.lower() == "urban100.zip":
                return p
        return cand[0]
    # fallback to a common name
    return search_dir / "Urban100.zip"

zip_urban_path = find_urban_zip(Path(testsets_root_path))

out_urban_hr_path  = Path(testsets_root_path) / "urban100/hr"
out_urban_lr4_path = Path(testsets_root_path) / "urban100/lr_x4"

for p in [out_urban_hr_path, out_urban_lr4_path]:
    p.mkdir(parents=True, exist_ok=True)

# ---- helper: unzip to a temp dir
def unzip_to_tmp_filetree(zip_file: Path, tmp_dir: Path):
    if not zip_file.exists():
        raise FileNotFoundError(f"missing zip: {zip_file}. put Urban100.zip in {testsets_root_path}")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(tmp_dir)
    return tmp_dir

# ---- helper: copy all images from a folder tree into dst
def copy_all_imgs_flat(src_root: Path, dst_dir: Path):
    cnt = 0
    for fp in src_root.rglob("*"):
        if fp.is_file() and fp.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".tif",".tiff"):
            # avoid duplicate names by prefixing parent if needed
            out_name = fp.name
            dst_path = dst_dir / out_name
            if dst_path.exists():
                out_name = f"{fp.parent.name}_{fp.name}"
                dst_path = dst_dir / out_name
            shutil.copy2(fp, dst_path)
            cnt += 1
    return cnt

# ---- helper: make lr_x4 from hr with bicubic; crop to be divisible by 4
def make_lr_x4_from_hr_dir(hr_dir: Path, lr_dir: Path):
    made = 0
    for f in sorted(hr_dir.glob("*")):
        if f.suffix.lower() not in (".png",".jpg",".jpeg",".bmp",".tif",".tiff"):
            continue
        img = Image.open(f).convert("RGB")
        w, h = img.size
        w4, h4 = w - (w % 4), h - (h % 4)
        if (w4, h4) != (w, h):
            img = img.crop((0, 0, w4, h4))
        lr = img.resize((img.width // 4, img.height // 4), resample=Image.BICUBIC)
        stem = f.stem
        if "x4" not in stem.lower():
            stem = stem + "x4"
        lr.save(lr_dir / f"{stem}.png")
        made += 1
    return made

# ---- 1) extract urban100 hr
tmp_urban_dir = Path("/content/tmp_urban_extract")
unzip_to_tmp_filetree(zip_urban_path, tmp_urban_dir)

# urban100 archives vary; grab images from the whole tree
n_urban_hr = copy_all_imgs_flat(tmp_urban_dir, out_urban_hr_path)

# ---- 2) make lr_x4
n_urban_lr = make_lr_x4_from_hr_dir(out_urban_hr_path, out_urban_lr4_path)

# ---- 3) sanity report
def count_imgs(d: Path):
    return len([f for f in d.glob("*") if f.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".tif",".tiff")])

def example_paths(dir_path: Path, k=3):
    return [str(p) for p in sorted(dir_path.glob('*.png'))[:k]]

print("done.")
print(f"urban100 hr: {count_imgs(out_urban_hr_path)}  | lr_x4: {count_imgs(out_urban_lr4_path)}  (expected ~100)")
print("\nexamples:")
print("urban100 lr_x4:", example_paths(out_urban_lr4_path))


done.
urban100 hr: 100  | lr_x4: 100  (expected ~100)

examples:
urban100 lr_x4: ['/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/urban100/lr_x4/img_001x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/urban100/lr_x4/img_002x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets/urban100/lr_x4/img_003x4.png']


In [ ]:
# counting train/val images and checking lr_x4 ↔ hr pairing


from pathlib import Path

# if you already defined these above, this block will just reuse them.
data_root = "/content/drive/MyDrive/dl_lab_super_resolution/data"
train_hr = f"{data_root}/div2k/hr"
train_lr = f"{data_root}/div2k/lr_x4"
train_hr_flickr = f"{data_root}/flickr2k/hr"
train_lr_flickr = f"{data_root}/flickr2k/lr_x4"
val_hr = f"{data_root}/div2k_official_val/hr"
val_lr = f"{data_root}/div2k_official_val/lr_x4"

def list_imgs(p):
    p = Path(p)
    if not p.exists(): return []
    return sorted([f for f in p.iterdir() if f.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".tif",".tiff")])

def count_imgs(p): return len(list_imgs(p))

def stem_set(p):
    # normalize by removing x4 suffix if present so names match
    return set([f.stem.lower().replace("x4","") for f in list_imgs(p)])

def report_split(name, hr_dir, lr4_dir):
    n_hr = count_imgs(hr_dir)
    n_lr = count_imgs(lr4_dir)
    s_hr = stem_set(hr_dir)
    s_lr = stem_set(lr4_dir)
    only_hr = sorted(list(s_hr - s_lr))[:10]
    only_lr = sorted(list(s_lr - s_hr))[:10]
    print(f"\n=== {name} ===")
    print(f"hr   : {n_hr}")
    print(f"lr_x4: {n_lr}")
    print(f"pair check x4 -> match: {len(s_hr & s_lr)} | only_hr: {len(s_hr - s_lr)} | only_lr: {len(s_lr - s_hr)}")
    if only_hr: print(f"  examples only in hr (first 10): {only_hr}")
    if only_lr: print(f"  examples only in lr (first 10): {only_lr}")
    if n_hr and n_lr:
        # show first 3 example paths
        ex_hr = [str(p) for p in list_imgs(hr_dir)[:3]]
        ex_lr = [str(p) for p in list_imgs(lr4_dir)[:3]]
        print("  sample hr :", ex_hr)
        print("  sample lr4:", ex_lr)

# train: div2k
report_split("div2k (train)", train_hr, train_lr)

# train: flickr2k
report_split("flickr2k (train)", train_hr_flickr, train_lr_flickr)

# validation: div2k_official_val
report_split("div2k_official_val (validation)", val_hr, val_lr)

# df2k aggregate hr count
div2k_hr_count = count_imgs(train_hr)
flickr2k_hr_count = count_imgs(train_hr_flickr)
print("\n=== df2k aggregate (hr only) ===")
print(f"div2k hr     : {div2k_hr_count}")
print(f"flickr2k hr  : {flickr2k_hr_count}")
print(f"df2k hr total: {div2k_hr_count + flickr2k_hr_count}  # expected ~3450 if full sets")



=== div2k (train) ===
hr   : 800
lr_x4: 800
pair check x4 -> match: 800 | only_hr: 0 | only_lr: 0
  sample hr : ['/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/hr/0001.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/hr/0002.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/hr/0003.png']
  sample lr4: ['/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/lr_x4/0001x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/lr_x4/0002x4.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/div2k/lr_x4/0003x4.png']

=== flickr2k (train) ===
hr   : 2650
lr_x4: 2650
pair check x4 -> match: 2650 | only_hr: 0 | only_lr: 0
  sample hr : ['/content/drive/MyDrive/dl_lab_super_resolution/data/flickr2k/hr/000001.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/flickr2k/hr/000002.png', '/content/drive/MyDrive/dl_lab_super_resolution/data/flickr2k/hr/000003.png']
  sample lr4: ['/content/drive/MyDrive/dl_lab_super_resolution/d

In [ ]:
# verify test-set pairs (hr <-> lr_x4) for set14, bsd100, urban100
# all-lowercase comments; concise summary; deterministic

from pathlib import Path

img_exts = (".png",".jpg",".jpeg",".bmp",".tif",".tiff")

def list_imgs(p: Path):
    return sorted([f for f in p.iterdir() if f.suffix.lower() in img_exts])

def normalize_stem(fp: Path):
    # drop trailing 'x4' in lr names so it matches hr stems
    return fp.stem.lower().replace("x4","")

def verify_split(name: str, hr_dir: Path, lr_dir: Path, show_examples: int = 3, show_mismatches: int = 5):
    if not hr_dir.exists() or not lr_dir.exists():
        print(f"\n=== {name} ===")
        print(f"missing directory -> hr: {hr_dir.exists()} lr_x4: {lr_dir.exists()}")
        return

    hr_files = list_imgs(hr_dir)
    lr_files = list_imgs(lr_dir)

    hr_keys = {normalize_stem(p): p for p in hr_files}
    lr_keys = {normalize_stem(p): p for p in lr_files}

    matched_keys = sorted(set(hr_keys) & set(lr_keys))
    only_hr_keys = sorted(set(hr_keys) - set(lr_keys))
    only_lr_keys = sorted(set(lr_keys) - set(hr_keys))

    print(f"\n=== {name} ===")
    print(f"hr count   : {len(hr_files)}")
    print(f"lr_x4 count: {len(lr_files)}")
    print(f"matched    : {len(matched_keys)}")
    print(f"only in hr : {len(only_hr_keys)}")
    print(f"only in lr : {len(only_lr_keys)}")

    if only_hr_keys:
        print("  examples only in hr :", only_hr_keys[:show_mismatches])
    if only_lr_keys:
        print("  examples only in lr :", only_lr_keys[:show_mismatches])

    if matched_keys:
        ex = matched_keys[:show_examples]
        ex_pairs = [(hr_keys[k].name, lr_keys[k].name) for k in ex]
        print("  sample matched pairs:", ex_pairs)

# ---- run over all test sets
root = Path("/content/drive/MyDrive/dl_lab_super_resolution/data/test_sets")
verify_split("Set14",    root/"set14/hr",    root/"set14/lr_x4")
verify_split("BSD100",   root/"bsd100/hr",   root/"bsd100/lr_x4")
verify_split("Urban100", root/"urban100/hr", root/"urban100/lr_x4")



=== Set14 ===
hr count   : 14
lr_x4 count: 14
matched    : 14
only in hr : 0
only in lr : 0
  sample matched pairs: [('baboon.png', 'baboonx4.png'), ('barbara.png', 'barbarax4.png'), ('bridge.png', 'bridgex4.png')]

=== BSD100 ===
hr count   : 100
lr_x4 count: 100
matched    : 100
only in hr : 0
only in lr : 0
  sample matched pairs: [('101085.png', '101085x4.png'), ('101087.png', '101087x4.png'), ('102061.png', '102061x4.png')]

=== Urban100 ===
hr count   : 100
lr_x4 count: 100
matched    : 100
only in hr : 0
only in lr : 0
  sample matched pairs: [('img_001.png', 'img_001x4.png'), ('img_002.png', 'img_002x4.png'), ('img_003.png', 'img_003x4.png')]


In [ ]:
# gpu + pytorch sanity, seeds, and fast defaults


import torch, os, random, numpy as np

def set_global_seed(unique_seed: int = 123):
    random.seed(unique_seed)
    np.random.seed(unique_seed)
    torch.manual_seed(unique_seed)
    torch.cuda.manual_seed_all(unique_seed)
    # deterministic for reproducibility (slower) — switch off if you want max speed
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def show_cuda_info():
    print("torch version :", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu name      :", torch.cuda.get_device_name(0))
        print("capability    :", torch.cuda.get_device_capability(0))
        print("total memory  : %.2f gb" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
        # a100 loves tf32 for speed in training; keep it off for eval metrics, on for training
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

set_global_seed(123)
show_cuda_info()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device        :", device)

# quick cuda test
x_test = torch.randn(1024, 1024, device=device)
y_test = x_test @ x_test.t()
torch.cuda.synchronize() if device.type == "cuda" else None
print("cuda test ok, tensor shape:", y_test.shape)


torch version : 2.8.0+cu126
cuda available: True
gpu name      : NVIDIA A100-SXM4-40GB
capability    : (8, 0)
total memory  : 39.56 gb
device        : cuda
cuda test ok, tensor shape: torch.Size([1024, 1024])


In [ ]:
# tiny dataset + loader for sr (paired lr/hr by stem); optimized for gpu feeding


from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

class paired_sr_dataset(Dataset):
    def __init__(self, hr_dir: str, lr_dir: str):
        self.hr_dir = Path(hr_dir)
        self.lr_dir = Path(lr_dir)
        self.hr_files = sorted([p for p in self.hr_dir.iterdir() if p.suffix.lower() in (".png",".jpg",".jpeg")])
        self.lr_map = {p.stem.lower().replace("x4",""): p for p in self.lr_dir.iterdir() if p.suffix.lower() in (".png",".jpg",".jpeg")}
        self.pairs = []
        for hrp in self.hr_files:
            k = hrp.stem.lower()
            if k in self.lr_map:
                self.pairs.append((self.lr_map[k], hrp))
        assert len(self.pairs) > 0, "no paired images found"

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        lr = Image.open(lr_path).convert("RGB")
        hr = Image.open(hr_path).convert("RGB")
        # to tensor in [0,1], chw
        lr_t = torch.from_numpy(np.array(lr)).permute(2,0,1).float() / 255.0
        hr_t = torch.from_numpy(np.array(hr)).permute(2,0,1).float() / 255.0
        return {"lr": lr_t, "hr": hr_t, "name": hr_path.stem}

def make_loader(hr_dir, lr_dir, batch_size=1, workers=2):
    ds = paired_sr_dataset(hr_dir, lr_dir)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

# example: create loaders for your val and tests (edit paths if needed)
data_root = "/content/drive/MyDrive/dl_lab_super_resolution/data"
ldr_val  = make_loader(f"{data_root}/div2k_official_val/hr", f"{data_root}/div2k_official_val/lr_x4")
ldr_s14  = make_loader(f"{data_root}/test_sets/set14/hr",    f"{data_root}/test_sets/set14/lr_x4")
ldr_b100 = make_loader(f"{data_root}/test_sets/bsd100/hr",   f"{data_root}/test_sets/bsd100/lr_x4")

print("val pairs :", len(ldr_val.dataset))
print("set14 pairs:", len(ldr_s14.dataset))
print("bsd100 pairs:", len(ldr_b100.dataset))


val pairs : 100
set14 pairs: 14
bsd100 pairs: 100


In [ ]:
#audit: check mod-4 + size + quick identity test (psnr on a small sample)
# audit training lr_x4 against hr for div2k and flickr2k
# checks: (a) hr divisible by 4 after mod-crop, (b) lr size == hr_mod/4
# also computes a quick psnr between saved lr and a re-bicubic'd lr on a small sample

from pathlib import Path
from PIL import Image
import numpy as np, math, random

img_exts = (".png",".jpg",".jpeg",".bmp",".tif",".tiff")

def list_imgs(p: Path):
    return sorted([f for f in p.iterdir() if f.suffix.lower() in img_exts])

def modcrop_wh(w: int, h: int, s: int = 4):
    return w - (w % s), h - (h % s)

def psnr(a: np.ndarray, b: np.ndarray):
    a = a.astype(np.float32); b = b.astype(np.float32)
    mse = np.mean((a - b) ** 2)
    if mse == 0: return 99.0
    return 20.0 * math.log10(255.0) - 10.0 * math.log10(mse)

def audit_split(name: str, hr_dir: str, lr_dir: str, sample_psnr_n: int = 12, seed: int = 123):
    hrp, lrp = Path(hr_dir), Path(lr_dir)
    hr_files = list_imgs(hrp)
    lr_map = {p.stem.lower().replace("x4",""): p for p in list_imgs(lrp)}
    bad_mod = []
    bad_size = []
    ok_pairs = []

    for hrf in hr_files:
        key = hrf.stem.lower()
        if key not in lr_map: continue
        lrf = lr_map[key]

        # hr mod-4 check
        with Image.open(hrf) as imh:
            w, h = imh.size
        w4, h4 = modcrop_wh(w, h, 4)
        if (w4 != w) or (h4 != h):
            # not a failure — we just note it's not already mod-4
            pass

        # lr size must equal hr_mod/4
        with Image.open(lrf) as iml:
            lw, lh = iml.size
        if (lw, lh) != (w4 // 4, h4 // 4):
            bad_size.append((hrf.name, (w, h), lrf.name, (lw, lh), (w4 // 4, h4 // 4)))
        else:
            ok_pairs.append((hrf, lrf, (w, h), (lw, lh)))

    # psnr sample: regenerate lr from hr and compare to saved lr
    random.seed(seed)
    sample = random.sample(ok_pairs, k=min(sample_psnr_n, len(ok_pairs)))
    psnrs = []
    for hrf, lrf, (w, h), (lw, lh) in sample:
        with Image.open(hrf).convert("RGB") as imh:
            if (w % 4) or (h % 4):
                imh = imh.crop((0, 0, w - (w % 4), h - (h % 4)))
            imlr = imh.resize((imh.width // 4, imh.height // 4), resample=Image.BICUBIC)
            arr_gen = np.array(imlr)
        with Image.open(lrf).convert("RGB") as iml_saved:
            arr_saved = np.array(iml_saved)
        if arr_gen.shape != arr_saved.shape:
            continue
        psnrs.append(psnr(arr_gen, arr_saved))

    print(f"\n=== {name} audit ===")
    print(f"hr files : {len(hr_files)}")
    print(f"lr files : {len(lr_map)}")
    print(f"size mismatches (lr != hr_mod/4): {len(bad_size)}")
    if bad_size:
        print("  example mismatch:", bad_size[0])
    if psnrs:
        print(f"psnr(saved_lr vs regenerated_lr) on {len(psnrs)} samples -> "
              f"mean: {np.mean(psnrs):.2f} dB | min: {np.min(psnrs):.2f} | max: {np.max(psnrs):.2f}")
        print("  note: matlab vs pillow bicubic can differ by ~0.05–0.2 dB; near-∞ means exact match.")

data_root = "/content/drive/MyDrive/dl_lab_super_resolution/data"
audit_split("div2k train x4",   f"{data_root}/div2k/hr",    f"{data_root}/div2k/lr_x4")
audit_split("flickr2k train x4",f"{data_root}/flickr2k/hr", f"{data_root}/flickr2k/lr_x4")



=== div2k train x4 audit ===
hr files : 800
lr files : 800
size mismatches (lr != hr_mod/4): 0
psnr(saved_lr vs regenerated_lr) on 12 samples -> mean: 56.20 dB | min: 55.39 | max: 57.24
  note: matlab vs pillow bicubic can differ by ~0.05–0.2 dB; near-∞ means exact match.

=== flickr2k train x4 audit ===
hr files : 2650
lr files : 2650
size mismatches (lr != hr_mod/4): 0
psnr(saved_lr vs regenerated_lr) on 12 samples -> mean: 99.00 dB | min: 99.00 | max: 99.00
  note: matlab vs pillow bicubic can differ by ~0.05–0.2 dB; near-∞ means exact match.
